# 07 · Fusión tardía, calibración y puntaje de riesgo 0–100 — Grupo 17 · Sprint 3
**Entorno CPU** (no requiere GPU: usa probabilidades ya calculadas). Decisión D7: el meta-clasificador se entrena en
**validación** (blending) y se evalúa en **test**; las ramas nunca vieron validación ni test.

| Variante | Entradas del meta-clasificador |
|---|---|
| V0 | Baseline TF-IDF+RL |
| V1 | DistilBERT |
| V2 | DistilBERT + URL (arquitectura de la Ficha) |
| V3 | DistilBERT + Baseline + URL |

Correos sin URL: valor neutro 0,5 + indicador `sin_url` (Ficha). Calibración: Platt o isotónica, elegida por Brier en validación cruzada.
**Punto de operación:** umbral de phishing elegido en validación con FPR ≤ 1 % (margen sobre la meta de 2 %), maximizando recall.
**Fuera de distribución:** si existen las probabilidades por fold del `06` v1.1, reentrena baseline y rama URL por fold y evalúa la fusión.

In [1]:
# Celda 1: Entorno y entradas
import os, sys, json, time, random, importlib.util
from pathlib import Path
import numpy as np, pandas as pd
SEED = 17; random.seed(SEED); np.random.seed(SEED); os.environ["PYTHONHASHSEED"] = str(SEED)
def detectar_entorno():
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"): return "kaggle"
    try:
        import google.colab  # noqa
        return "colab"
    except ImportError:
        return "local"
ENV = detectar_entorno()
if ENV == "colab":
    from google.colab import drive; drive.mount("/content/drive"); BASE = Path("/content/drive/MyDrive/PI_G17")
elif ENV == "kaggle":
    BASE = Path("/kaggle/working/PI_G17")
else:
    BASE = Path(os.environ.get("PI_G17_BASE", Path.cwd() / "PI_G17"))
PROC, MODELS, RES = BASE / "data/processed", BASE / "models", BASE / "results"
def cargar(nombre, ruta):
    spec = importlib.util.spec_from_file_location(nombre, ruta); m = importlib.util.module_from_spec(spec)
    sys.modules[nombre] = m; spec.loader.exec_module(m); return m
ioc = cargar("ioc_utils", BASE / "src/ioc_utils.py"); uf = cargar("url_features", BASE / "src/url_features.py")
for f in ("emails_v1.parquet", "url_scores_v1.parquet", "text_scores_distilbert_v1.parquet"):
    assert (PROC / f).exists(), f"Falta {f}: ejecutar 03/04/06 antes"
assert (MODELS / "baseline_tfidf_lr_v1.joblib").exists(), "Falta el baseline (05)"
D = pd.read_parquet(PROC / "emails_v1.parquet")
U = pd.read_parquet(PROC / "url_scores_v1.parquet")[["id", "url_b_max", "url_b_media", "n_urls_eval"]]
T = pd.read_parquet(PROC / "text_scores_distilbert_v1.parquet")
CLASES = ["legitimo", "spam", "phishing"]; PH = 2
print(ENV, "| correos", len(D), "| prob. DistilBERT", len(T))

Mounted at /content/drive
colab | correos 54684 | prob. DistilBERT 16406


In [2]:
# Celda 2: Métricas, umbral por FPR, calibración y features
from sklearn.metrics import f1_score, precision_recall_fscore_support, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import StratifiedKFold, cross_val_predict
def metricas(y, pred):
    p, r, f, _ = precision_recall_fscore_support(y, pred, labels=[0, 1, 2], zero_division=0)
    no_ph = y != PH
    return {"f1_macro": round(f1_score(y, pred, average="macro", labels=[0, 1, 2], zero_division=0), 4),
            "precision": dict(zip(CLASES, np.round(p, 4).tolist())), "recall": dict(zip(CLASES, np.round(r, 4).tolist())),
            "fpr_phishing": round(float((pred[no_ph] == PH).mean()), 4) if no_ph.any() else None,
            "fpr_legitimo_a_phishing": round(float((pred[y == 0] == PH).mean()), 4) if (y == 0).any() else None, "n": int(len(y))}
def umbral_por_fpr(y, p_ph, fpr_max=0.01):
    """Umbral más bajo (máximo recall) cuyo FPR en validación no supera fpr_max."""
    neg = np.sort(p_ph[y != PH])[::-1]; k = int(np.floor(fpr_max * len(neg)))
    return float(np.nextafter(neg[k], 1)) if k < len(neg) else 0.0
def predecir(P, u):
    pred = P[:, :2].argmax(1); pred[P[:, PH] >= u] = PH; return pred
def brier(y, P): return round(float(np.mean(np.sum((P - np.eye(3)[y]) ** 2, 1))), 5)
def ece_phishing(y, p, bins=10):
    yb = (y == PH).astype(float); e = 0.0
    for lo in np.linspace(0, 1, bins, endpoint=False):
        m = (p >= lo) & (p < lo + 1 / bins) if lo < 1 - 1 / bins else (p >= lo)
        if m.any(): e += m.mean() * abs(yb[m].mean() - p[m].mean())
    return round(float(e), 5)
lg = lambda P: np.log(np.clip(P, 1e-6, 1.0))
def features(variante, Pd, Pb, url):
    """Pd/Pb: probabilidades (n,3) de DistilBERT/baseline; url: DataFrame con url_b_max, url_b_media, n_urls_eval."""
    sin = (url["n_urls_eval"].fillna(0).values == 0).astype(float)
    bloque_url = np.c_[url["url_b_max"].fillna(0.5).values, url["url_b_media"].fillna(0.5).values, sin, np.log1p(url["n_urls_eval"].fillna(0).values)]
    return {"V0": lg(Pb), "V1": lg(Pd), "V2": np.c_[lg(Pd), bloque_url], "V3": np.c_[lg(Pd), lg(Pb), bloque_url]}[variante]
def meta(metodo):
    return CalibratedClassifierCV(LogisticRegression(C=1.0, max_iter=3000), method=metodo, cv=5)
def evaluar_variante(variante, Xv, yv, Xt, yt):
    """Elige calibración por Brier en CV sobre val; umbral FPR≤1 % sobre probabilidades CV de val; ajusta en val y evalúa en test."""
    cv = StratifiedKFold(5, shuffle=True, random_state=SEED); mejor = None
    for metodo in ("sigmoid", "isotonic"):
        Pcv = cross_val_predict(meta(metodo), Xv, yv, cv=cv, method="predict_proba")
        b = brier(yv, Pcv)
        if mejor is None or b < mejor[1]: mejor = (metodo, b, Pcv)
    metodo, b_val, Pcv = mejor
    u = umbral_por_fpr(yv, Pcv[:, PH]); m = meta(metodo).fit(Xv, yv); Pt = m.predict_proba(Xt)
    return m, u, Pt, {"calibracion": metodo, "brier_val_cv": b_val, "umbral_fpr1": round(u, 5),
                      "val_cv_umbral": metricas(yv, predecir(Pcv, u)), "test_umbral": metricas(yt, predecir(Pt, u)),
                      "test_argmax": metricas(yt, Pt.argmax(1)), "brier_test": brier(yt, Pt), "ece_phishing_test": ece_phishing(yt, Pt[:, PH])}

In [3]:
# Celda 3: Probabilidades del baseline en val/test
import joblib
base = joblib.load(MODELS / "baseline_tfidf_lr_v1.joblib")
V = D[D["split"].isin(["val", "test"])].merge(T[["id", "p_legitimo", "p_spam", "p_phishing"]], on="id", how="inner").merge(U, on="id", how="left")
assert len(V) == (D["split"].isin(["val", "test"])).sum(), "Las probabilidades de DistilBERT no cubren val+test"
t0 = time.time()
Pb_all = base.predict_proba(V["texto_modelo"].str[:5000])
Pd_all = V[["p_legitimo", "p_spam", "p_phishing"]].values
mv, mt = (V["split"] == "val").values, (V["split"] == "test").values
yv, yt = V.loc[mv, "y"].values, V.loc[mt, "y"].values
print(f"baseline en val+test: {time.time()-t0:.0f}s | val {mv.sum()} | test {mt.sum()}")

baseline en val+test: 32s | val 8203 | test 8203


In [4]:
# Celda 4: Las 4 variantes en la partición principal
RES_F, MODELOS = {}, {}
for v in ("V0", "V1", "V2", "V3"):
    X = features(v, Pd_all, Pb_all, V)
    m, u, Pt, r = evaluar_variante(v, X[mv], yv, X[mt], yt)
    RES_F[v], MODELOS[v] = r, (m, u, Pt)
    print(v, r["calibracion"], "| test recall ph", r["test_umbral"]["recall"]["phishing"], "| FPR", r["test_umbral"]["fpr_phishing"],
          "| F1", r["test_umbral"]["f1_macro"], "| Brier", r["brier_test"], "| ECE", r["ece_phishing_test"])
# Selección: mayor recall de phishing en val-CV con FPR≤1 %; desempate por Brier. Se reporta igual que las demás.
ELEGIDA = max(RES_F, key=lambda v: (RES_F[v]["val_cv_umbral"]["recall"]["phishing"], -RES_F[v]["brier_val_cv"]))
m, u, Pt = MODELOS[ELEGIDA]
riesgo = np.round(100 * Pt[:, PH]).astype(int)
RES_F["elegida"] = ELEGIDA
RES_F["elegida_detalle"] = {"matriz_confusion_test": confusion_matrix(yt, predecir(Pt, u), labels=[0, 1, 2]).tolist(),
    "recall_phishing_por_grupo": {g: round(float((predecir(Pt, u)[(V.loc[mt, "grupo"] == g).values] == PH).mean()), 4)
                                  for g in sorted(V.loc[mt & (V["clase"] == "phishing").values, "grupo"].unique())},
    "riesgo_mediana_por_clase": {c: int(np.median(riesgo[yt == k])) for k, c in enumerate(CLASES)},
    "pct_phishing_con_riesgo_>=90": round(100 * float((riesgo[yt == PH] >= 90).mean()), 1),
    "pct_legitimo_con_riesgo_>=50": round(100 * float((riesgo[yt == 0] >= 50).mean()), 2)}
joblib.dump({"modelo": m, "umbral_fpr1": u, "variante": ELEGIDA}, MODELS / "fusion_v1.joblib")
(MODELS / "fusion_v1.meta.json").write_text(json.dumps({"variante": ELEGIDA, "umbral_phishing": u, "calibracion": RES_F[ELEGIDA]["calibracion"],
    "features": "log(P DistilBERT)[3] (+ log(P baseline)[3] en V3) + url_b_max, url_b_media (NaN->0.5), sin_url, log1p(n_urls)",
    "riesgo": "round(100 * P(phishing) calibrada)", "semilla": SEED}, indent=1, ensure_ascii=False))
print("ELEGIDA:", ELEGIDA, RES_F["elegida_detalle"])

V0 isotonic | test recall ph 0.9872 | FPR 0.0109 | F1 0.9816 | Brier 0.02486 | ECE 0.01
V1 isotonic | test recall ph 0.9953 | FPR 0.0121 | F1 0.9842 | Brier 0.01396 | ECE 0.00335
V2 isotonic | test recall ph 0.9966 | FPR 0.0103 | F1 0.9862 | Brier 0.01261 | ECE 0.00196
V3 isotonic | test recall ph 0.9966 | FPR 0.0091 | F1 0.9874 | Brier 0.01109 | ECE 0.00548
ELEGIDA: V3 {'matriz_confusion_test': [[3196, 4, 7], [17, 3437, 54], [3, 2, 1483]], 'recall_phishing_por_grupo': {'Nigerian_Fraud': 0.9939, 'nazario_anual': 1.0, 'phishing_pot': 0.9964}, 'riesgo_mediana_por_clase': {'legitimo': 0, 'spam': 0, 'phishing': 100}, 'pct_phishing_con_riesgo_>=90': 94.3, 'pct_legitimo_con_riesgo_>=50': 0.0}


In [5]:
# Celda 5: Fusión fuera de distribución (requiere 06 v1.1)
import xgboost as xgb
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
FOLDS = ["loso_phishing_pot", "loso_Nigerian_Fraud", "loso_nazario_anual", "temporal_phishing_2025+"]
disponibles = [f for f in FOLDS if (PROC / f"text_scores_distilbert_v1_{f}.parquet").exists()]
OOD = {f: "pendiente: ejecutar 06 v1.1" for f in FOLDS if f not in disponibles}
if disponibles:
    Dfull = pd.read_parquet(PROC / "emails_v1.parquet", columns=["id", "urls_defang"])
    filas = [(i, ioc.refang(u)) for i, l in zip(Dfull["id"], Dfull["urls_defang"]) for u in list(l)[:20]]
    UU = pd.DataFrame(filas, columns=["id", "url"])                     # vivas SOLO en memoria
    FU = pd.DataFrame([uf.features_url(x) for x in UU["url"]])
    UU["dominio"] = [".".join(uf.descomponer(x)[0].split(".")[-2:]) for x in UU["url"]]; UU = UU.drop(columns="url")
    def pipeline_base():
        feats = FeatureUnion([("p", TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.9, max_features=100_000, sublinear_tf=True)),
                              ("c", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=5, max_features=100_000, sublinear_tf=True))])
        return Pipeline([("tfidf", feats), ("lr", LogisticRegression(C=8.0, max_iter=2000, class_weight="balanced"))])
for f in disponibles:
    t0 = time.time(); s = D[f"fold_{f}"]; fuera = D[f"fuera_{f}"].values
    tr, va, te = (s == "train").values, (s == "val").values, (s == "test").values
    # Rama URL del fold: sin los dominios de val/test del fold (y por tanto sin los del grupo fuera)
    ids_tr, ids_eval = set(D.loc[tr, "id"]), set(D.loc[va | te, "id"])
    dom_eval = set(UU.loc[UU["id"].isin(ids_eval), "dominio"])
    yU = UU["id"].map(dict(zip(D["id"], (D["clase"] == "phishing").astype(int))))
    mtr = UU["id"].isin(ids_tr).values & ~UU["dominio"].isin(dom_eval).values
    mva = UU["id"].isin(set(D.loc[va, "id"])).values
    top = FU.loc[mtr, "tld"].value_counts().head(50).index.tolist()
    XU = FU[uf.NUMERICAS].astype(float).copy()
    XU["tld"] = pd.Categorical(FU["tld"].where(FU["tld"].isin(top), "__otro__"), categories=top + ["__otro__"])
    mx = xgb.XGBClassifier(n_estimators=1000, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8, tree_method="hist",
                           enable_categorical=True, eval_metric="aucpr", early_stopping_rounds=50, random_state=SEED, n_jobs=-1)
    mx.fit(XU[mtr], yU[mtr], eval_set=[(XU[mva], yU[mva])], verbose=False)
    sU = pd.Series(mx.predict_proba(XU)[:, 1], index=UU.index)
    urlf = sU.groupby(UU["id"]).agg(url_b_max="max", url_b_media="mean").join(UU.groupby("id").size().rename("n_urls_eval"))
    # Baseline del fold
    bf = pipeline_base().fit(D.loc[tr, "texto_modelo"].str[:5000], D.loc[tr, "y"])
    # DistilBERT del fold (06 v1.1)
    Tf = pd.read_parquet(PROC / f"text_scores_distilbert_v1_{f}.parquet")
    W = D.loc[va | te, ["id", "y", "texto_modelo"]].assign(split=np.where(va[va | te], "val", "test"), fuera=fuera[va | te])
    W = W.merge(Tf[["id", "p_legitimo", "p_spam", "p_phishing"]], on="id").merge(urlf.reset_index(), on="id", how="left")
    Pb = bf.predict_proba(W["texto_modelo"].str[:5000]); Pd = W[["p_legitimo", "p_spam", "p_phishing"]].values
    wv, wt = (W["split"] == "val").values, (W["split"] == "test").values; ft = W.loc[wt, "fuera"].values
    OOD[f] = {"segundos": None}
    for v in ("V0", "V1", "V2", "V3"):
        X = features(v, Pd, Pb, W)
        _, u_, Pt_, r = evaluar_variante(v, X[wv], W.loc[wv, "y"].values, X[wt], W.loc[wt, "y"].values)
        OOD[f][v] = {"recall_phishing_fuera": round(float((predecir(Pt_, u_)[ft] == PH).mean()), 4),
                     "fpr_phishing_test": r["test_umbral"]["fpr_phishing"], "f1_test": r["test_umbral"]["f1_macro"]}
    OOD[f]["segundos"] = round(time.time() - t0)
    print(f, {v: OOD[f][v]["recall_phishing_fuera"] for v in ("V0", "V1", "V2", "V3")}, f'{OOD[f]["segundos"]}s')

loso_phishing_pot {'V0': 0.6848, 'V1': 0.7364, 'V2': 0.8352, 'V3': 0.8352} 169s
loso_Nigerian_Fraud {'V0': 0.917, 'V1': 0.8483, 'V2': 0.6781, 'V3': 0.941} 188s
loso_nazario_anual {'V0': 0.9552, 'V1': 0.9634, 'V2': 0.9634, 'V3': 0.9744} 177s
temporal_phishing_2025+ {'V0': 0.9333, 'V1': 0.9412, 'V2': 0.9794, 'V3': 0.9858} 164s


In [6]:
# Celda 6: Guardar y RESUMEN
R = {"entorno": ENV, "principal": RES_F, "fuera_de_distribucion": OOD}
(RES / "fusion_v1.json").write_text(json.dumps(R, indent=1, ensure_ascii=False, default=str))
corto = {v: {"recall_ph": RES_F[v]["test_umbral"]["recall"]["phishing"], "fpr_ph": RES_F[v]["test_umbral"]["fpr_phishing"],
             "f1": RES_F[v]["test_umbral"]["f1_macro"], "brier": RES_F[v]["brier_test"], "ece": RES_F[v]["ece_phishing_test"],
             "calib": RES_F[v]["calibracion"]} for v in ("V0", "V1", "V2", "V3")}
print("===== RESUMEN (07) =====")
print(json.dumps({"entorno": ENV, "test_por_variante": corto, "elegida": RES_F["elegida"], "elegida_detalle": RES_F["elegida_detalle"],
                  "fuera_de_distribucion": OOD}, ensure_ascii=False, default=str, separators=(",", ":")))

===== RESUMEN (07) =====
{"entorno":"colab","test_por_variante":{"V0":{"recall_ph":0.9872,"fpr_ph":0.0109,"f1":0.9816,"brier":0.02486,"ece":0.01,"calib":"isotonic"},"V1":{"recall_ph":0.9953,"fpr_ph":0.0121,"f1":0.9842,"brier":0.01396,"ece":0.00335,"calib":"isotonic"},"V2":{"recall_ph":0.9966,"fpr_ph":0.0103,"f1":0.9862,"brier":0.01261,"ece":0.00196,"calib":"isotonic"},"V3":{"recall_ph":0.9966,"fpr_ph":0.0091,"f1":0.9874,"brier":0.01109,"ece":0.00548,"calib":"isotonic"}},"elegida":"V3","elegida_detalle":{"matriz_confusion_test":[[3196,4,7],[17,3437,54],[3,2,1483]],"recall_phishing_por_grupo":{"Nigerian_Fraud":0.9939,"nazario_anual":1.0,"phishing_pot":0.9964},"riesgo_mediana_por_clase":{"legitimo":0,"spam":0,"phishing":100},"pct_phishing_con_riesgo_>=90":94.3,"pct_legitimo_con_riesgo_>=50":0.0},"fuera_de_distribucion":{"loso_phishing_pot":{"segundos":169,"V0":{"recall_phishing_fuera":0.6848,"fpr_phishing_test":0.0124,"f1_test":0.888},"V1":{"recall_phishing_fuera":0.7364,"fpr_phishing_tes